# Customer Segmentation Analysis

This notebook performs customer segmentation analysis for the SOKO e-commerce platform using various clustering techniques.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

In [ ]:
# Load customer data
# In a real scenario, this would come from the data warehouse
customer_data = pd.DataFrame({
    'customer_id': range(1, 101),
    'total_orders': np.random.poisson(3, 100),
    'total_spent': np.random.exponential(200, 100),
    'avg_order_value': np.random.normal(75, 25, 100),
    'days_since_last_order': np.random.exponential(30, 100),
    'product_categories_purchased': np.random.randint(1, 8, 100)
})

print(f"Dataset shape: {customer_data.shape}")
print(customer_data.head())

In [ ]:
# Data preprocessing
# Handle missing values
customer_data = customer_data.dropna()

# Select features for clustering
features = ['total_orders', 'total_spent', 'avg_order_value', 'days_since_last_order', 'product_categories_purchased']
X = customer_data[features]

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features standardized successfully")

In [ ]:
# Determine optimal number of clusters using Elbow method
inertia = []
silhouette_scores = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)
    
    if k > 1:
        silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

# Plot Elbow curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(k_range, inertia, 'bo-')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method')
ax1.grid(True)

ax2.plot(k_range[1:], silhouette_scores, 'ro-')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Analysis')
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Perform K-means clustering with optimal k
optimal_k = 4  # Based on elbow method and silhouette analysis
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
customer_data['cluster'] = kmeans.fit_predict(X_scaled)

print(f"Clustering completed with {optimal_k} clusters")
print(customer_data['cluster'].value_counts().sort_index())

In [ ]:
# Analyze cluster characteristics
cluster_summary = customer_data.groupby('cluster')[features].agg(['mean', 'std', 'min', 'max'])
print("Cluster Summary Statistics:")
print(cluster_summary)

In [ ]:
# Visualize clusters using PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=customer_data['cluster'], 
                     cmap='viridis', alpha=0.6, s=50)

# Plot cluster centers
centers_pca = pca.transform(kmeans.cluster_centers_)
plt.scatter(centers_pca[:, 0], centers_pca[:, 1], c='red', marker='X', s=200, 
           linewidth=3, edgecolors='black', label='Cluster Centers')

plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.title('Customer Segments (PCA Projection)')
plt.colorbar(scatter, label='Cluster')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Explained variance ratio: {pca.explained_variance_ratio_}")

In [ ]:
# Create detailed cluster profiles
def create_cluster_profile(cluster_data, cluster_num):
    profile = {
        'cluster': cluster_num,
        'size': len(cluster_data),
        'percentage': len(cluster_data) / len(customer_data) * 100,
        'avg_total_orders': cluster_data['total_orders'].mean(),
        'avg_total_spent': cluster_data['total_spent'].mean(),
        'avg_order_value': cluster_data['avg_order_value'].mean(),
        'avg_days_since_last_order': cluster_data['days_since_last_order'].mean(),
        'avg_categories': cluster_data['product_categories_purchased'].mean()
    }
    return profile

cluster_profiles = []
for cluster in range(optimal_k):
    cluster_data = customer_data[customer_data['cluster'] == cluster]
    profile = create_cluster_profile(cluster_data, cluster)
    cluster_profiles.append(profile)

profiles_df = pd.DataFrame(cluster_profiles)
print("Cluster Profiles:")
print(profiles_df.round(2))

In [ ]:
# Visualize feature distributions by cluster
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for i, feature in enumerate(features):
    if i < len(axes):
        sns.boxplot(data=customer_data, x='cluster', y=feature, ax=axes[i])
        axes[i].set_title(f'{feature.replace("_", " ").title()} by Cluster')
        axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Assign meaningful names to clusters based on characteristics
cluster_names = {
    0: 'High-Value Frequent Buyers',
    1: 'Moderate Spenders',
    2: 'Low-Value Occasional Buyers',
    3: 'New Customers'
}

customer_data['segment_name'] = customer_data['cluster'].map(cluster_names)

# Display segment distribution
segment_distribution = customer_data['segment_name'].value_counts()
plt.figure(figsize=(10, 6))
segment_distribution.plot(kind='bar', color='skyblue')
plt.title('Customer Segment Distribution')
plt.xlabel('Segment')
plt.ylabel('Number of Customers')
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3)
plt.show()

print("Segment Distribution:")
print(segment_distribution)

In [ ]:
# Generate insights and recommendations
print("\n=== CUSTOMER SEGMENTATION INSIGHTS ===\n")

for cluster, name in cluster_names.items():
    profile = profiles_df[profiles_df['cluster'] == cluster].iloc[0]
    print(f"{name} (Cluster {cluster}):")
    print(f"  - Size: {profile['size']:.0f} customers ({profile['percentage']:.1f}%)")
    print(f"  - Avg Orders: {profile['avg_total_orders']:.1f}")
    print(f"  - Avg Spend: ${profile['avg_total_spent']:.2f}")
    print(f"  - Avg Order Value: ${profile['avg_order_value']:.2f}")
    print(f"  - Recency: {profile['avg_days_since_last_order']:.1f} days")
    print()

print("RECOMMENDATIONS:")
print("1. High-Value Frequent Buyers: Implement VIP program with exclusive offers")
print("2. Moderate Spenders: Send personalized recommendations and bundle deals")
print("3. Low-Value Occasional Buyers: Focus on re-engagement campaigns")
print("4. New Customers: Provide onboarding discounts and educational content")

In [ ]:
# Save results
output_file = 'customer_segments.csv'
customer_data.to_csv(output_file, index=False)
print(f"\nResults saved to {output_file}")

# Save cluster profiles
profiles_file = 'cluster_profiles.csv'
profiles_df.to_csv(profiles_file, index=False)
print(f"Cluster profiles saved to {profiles_file}")